<a href="https://colab.research.google.com/github/MariamWassim/internship-/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MariamWassim/internship-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane: CTR / Engagement Opportunity Scoring.** Which visible pages under-capture clicks for their position, and deserve a metadata/content review?

> Skills loaded for this notebook: `writing-data-contracts/SKILL.md` + `flyrank/flyrank-data/SKILL.md`.


## 1. Unit of analysis + time window

**Source row** (before I aggregate anything): one row in `fact_content_daily_performance` = one `report_date` × `client_hash_id` × `content_hash_id` — one page, one client, one day. I'm reading only the `month=2026-03` partition (mid-panel month, per the panel warning — never `_sample`, which is June 2026, the panel's final month and therefore a sealed test month for any past→future label).

**My analysis unit** (what I actually score and rank): one row = one page for one client, **aggregated across all of March 2026** — i.e. `(client_hash_id, content_hash_id)` → one row per page-month. I collapse the daily grain into a monthly grain because the decision this supports ("which pages should a reviewer look at first") is made once, not once per day.

**Time window:** `report_date` between `2026-03-01` and `2026-03-31` inclusive.

**What I'd rank (not a binary label — this is a scoring/ranking lane, not classification):** a CTR-gap opportunity score per page — actual March CTR minus the *expected* CTR for that page's position tier (computed from its peers in the same tier, same month). Pages sitting furthest below their tier's expected CTR, with enough impression volume to not be noise, rank highest — they're pages Google already shows, that aren't earning the clicks a page in that position normally earns.

**One thing I deliberately exclude:** rows where `gsc_avg_position = 0`. Per the data skill, `0` means *"no position data,"* not "ranked in position zero" — keeping them would corrupt the position-tier grouping by dumping true rank-1 pages and no-data pages into the same bucket.


In [ ]:
# 1a. Connect DuckDB to the release. Token comes from the Colab Secret panel (key icon, left sidebar),
# named HF_TOKEN — never pasted directly into a cell (this repo is public).
%pip -q install duckdb

import os
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Sanity check BEFORE trusting any column name I haven't personally confirmed against this table:
con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT} LIMIT 1").df()


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` (summed over March) | Feature | Known the moment March's search data lands — nothing about April involved. |
| `gsc_clicks` (summed over March) | Feature — **but see the trap in section 3** | Known at decision time, but it is *inside the label's own formula* (`ctr = clicks / impressions`), so it can't be used as an ordinary predictor once I define an opportunity label from CTR. |
| `gsc_avg_position` (averaged over March, real values only) | Feature | Known at decision time; used to build the position tier. |
| `position_tier` (derived from `gsc_avg_position`) | Feature | Deterministic bucket of an already-knowable feature. |
| `ctr_mar` = clicks/impressions for March | **Label / proxy source** | This is what the opportunity score is built from — the target of the ranking, not an input to it. |
| `expected_ctr_tier` (peer mean CTR within the same tier, page excluded) | Feature | Computed only from other pages' March data — available at the same moment. |
| `client_hash_id`, `content_hash_id`, `report_date`/`month` | Context | Grouping and joining only — pseudonyms, never model inputs. |
| `gsc_avg_position = 0` rows | Excluded | Means "no position data," not rank zero — would corrupt the tier grouping. |
| `fact_content_daily_performance_sample` (the whole table) | Excluded | It's June 2026, the panel's final month — the natural outcome window for any past→future label. Never touched for label logic. |
| Any FlyRank product decision fields (`health_score`, `priority_score`, `action_type`) | Excluded | Not shipped in this dataset by design — and if I ever rebuilt one, the lane guide is explicit it must never be a model feature. |


In [ ]:
# No query needed here beyond what's already run above — this section is the classification table itself.
# (Kept as a code cell per the skeleton's structure; nothing to execute.)
print('Field classification is documented above.')


## 3. Verify it with queries (grain, counts, missing values, windows)


In [ ]:
# --- Query A: GRAIN — one row really is one (report_date, client, content) ---
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print('Rows violating the stated grain (should be EMPTY):')
print(grain_check)

# --- Query B: row count + date span for my slice ---
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {FACT_MAR}
""").df()
print('\nMarch 2026 slice — row count and date span:')
print(counts)

# --- Query C: AVAILABILITY, filtered with IS TRUE as required ---
avail = con.sql(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN gsc_avg_position IS NOT NULL AND gsc_avg_position > 0 THEN 1 ELSE 0 END) AS n_real_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS n_ga4_available
    FROM {FACT_MAR}
""").df()
print('\nAvailability — how many rows survive the IS TRUE filter:')
print(avail)
print(f"\nGA4-available share: {avail['n_ga4_available'][0] / avail['n_total'][0]:.1%}")


## 4. Five features + the leakage trap

Building the page-month feature frame, then five features — each with an "available at the decision moment because…" line.


In [ ]:
# Aggregate the daily grain up to my analysis grain: one row per (client, content) for March.
page_month = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_mar,
        SUM(gsc_clicks) AS clicks_mar,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_mar
    FROM {FACT_MAR}
    GROUP BY 1, 2
    HAVING impressions_mar >= 100          -- min-volume filter: keep noise out
       AND avg_position_mar IS NOT NULL    -- drop pages with only position=0 (no-data) rows
""").df()

page_month['ctr_mar'] = page_month['clicks_mar'] / page_month['impressions_mar']

def position_tier(p):
    if p <= 3: return '1-3'
    if p <= 10: return '4-10'
    if p <= 20: return '11-20'
    return '21+'

page_month['position_tier'] = page_month['avg_position_mar'].apply(position_tier)
tier_order = {'1-3': 0, '4-10': 1, '11-20': 2, '21+': 3}
page_month['position_tier_num'] = page_month['position_tier'].map(tier_order)

# expected CTR for a page's tier = mean CTR of ITS PEERS in the same tier (page's own value excluded)
tier_mean = page_month.groupby('position_tier')['ctr_mar'].transform('mean')
tier_n = page_month.groupby('position_tier')['ctr_mar'].transform('count')
# leave-one-out mean so a page never gets compared against a tier average that includes itself
page_month['expected_ctr_tier'] = (tier_mean * tier_n - page_month['ctr_mar']) / (tier_n - 1)

page_month['ctr_gap'] = page_month['expected_ctr_tier'] - page_month['ctr_mar']

print(f'{len(page_month):,} pages with enough March volume to score')
page_month.sort_values('ctr_gap', ascending=False).head(10)


In [ ]:
# Five features, each with an availability line:

feature_notes = {
    'impressions_mar':   'Available because it is the sum of March daily impressions only — all observed before the review moment, nothing from April onward.',
    'avg_position_mar':  'Available because it is averaged only over March daily position readings for this page — no future data involved.',
    'position_tier_num': 'Available because it is a deterministic bucket of avg_position_mar — derived, not predicted.',
    'ctr_mar':           'Available because clicks and impressions are both already-observed March totals — but see the trap below: it is the SOURCE of the label, not an ordinary feature.',
    'expected_ctr_tier': 'Available because it only uses OTHER pages\' March CTR in the same tier — computed at the same moment, no peeking forward.',
}
for feat, note in feature_notes.items():
    print(f'- {feat}: {note}')


### The trap (deliberate leak, on purpose)

Define a binary **opportunity label**: is this page under-clicking for its tier, with enough volume to matter?
`is_opportunity = (ctr_mar < expected_ctr_tier) AND (impressions_mar >= 100)`.

First: an **honest quick score**, trained only on features that don't touch the label's own ingredients.
Then: the **trap** — add `clicks_mar` as a "feature." It's literally one of the two numbers `ctr_mar` is computed from, and `ctr_mar` is what the label is built from. Watch the score jump toward perfect. Then delete it and keep the honest number.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

scored = page_month.dropna(subset=['avg_position_mar', 'ctr_mar', 'expected_ctr_tier']).copy()
scored['is_opportunity'] = (
    (scored['ctr_mar'] < scored['expected_ctr_tier']) & (scored['impressions_mar'] >= 100)
).astype(int)

print('Opportunity rate:', scored['is_opportunity'].mean().round(3))

# --- HONEST quick score: no feature derived from clicks/ctr ---
honest_features = ['impressions_mar', 'avg_position_mar', 'position_tier_num']
X, y = scored[honest_features], scored['is_opportunity']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f'\nHONEST quick score (AUC): {honest_auc:.3f}')

# --- THE TRAP: add clicks_mar, which is literally inside ctr_mar's own formula ---
leaky_features = honest_features + ['clicks_mar']
Xl, yl = scored[leaky_features], scored['is_opportunity']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.3, random_state=42, stratify=yl)
leaky_model = LogisticRegression(max_iter=1000).fit(Xl_tr, yl_tr)
leaky_auc = roc_auc_score(yl_te, leaky_model.predict_proba(Xl_te)[:, 1])
print(f'LEAKY quick score (AUC), with clicks_mar added: {leaky_auc:.3f}  <-- watch this jump toward 1.0')

# --- delete the leak, keep the honest number ---
print(f'\nclicks_mar dropped for good — it is inside the label formula, not a real predictor.')
print(f'KEEPING the honest AUC: {honest_auc:.3f}')


## 5. Data limits

**Named limitation: this slice is not a level playing field across clients.** History depth is a wildly unbalanced panel — some clients have 12+ months of GSC history behind them by March 2026, others started tracking much later or not at all before this month (checkable against `dim_clients.gsc_data_start`). A client whose tracking only started in February shows up in this March slice with a thin, noisy CTR baseline compared to a client with a year of history — my opportunity ranking can quietly overweight noise from newer clients unless I add a per-client minimum-history filter before trusting the ranking across clients.

**A second limitation, worth naming honestly:** this entire analysis only ever sees pages Google is *already showing* (they have real `gsc_avg_position` and impressions). A page with strong content that has never earned any search visibility literally cannot appear in this ranking at all — so "no opportunities found" for a client could mean "nothing to fix" or could mean "this client barely has any visible pages yet." This method can only ever say *improve what's already showing*, never *here's content nobody has found yet*.


In [ ]:
# Confirm the client-history limitation with a real number before asserting it in prose:
history_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        SUM(CASE WHEN gsc_data_start >= DATE '2026-02-01' THEN 1 ELSE 0 END) AS n_thin_history_clients
    FROM {DIM_CLIENTS}
""").df()
print(history_check)
print(f"Share of clients with under a month of GSC history before this slice: "
      f"{history_check['n_thin_history_clients'][0] / history_check['n_clients'][0]:.1%}")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
